# Frozen DINO Similarity Retrieval baseline - Stanford Cars

Notebook này triển khai method thứ 3 trong Table 2 của paper: **Frozen DINO Similarity Ret.**

Bản chất method theo paper: tạo candidate pool từ 20% train split, dùng DINOv3 pretrained/frozen để tính embedding cho query và candidate, chọn context image gần nhất theo similarity, rồi train downstream classifier trên cặp `(query image, retrieved context image)`. Retriever là frozen và không học; chỉ downstream multi-input classifier được fine-tune.

Notebook giữ kiến trúc đầu giống notebook Random Context Ret. bạn đã chỉnh: mount Drive, unzip dataset, tải DINO weight, cấu hình output, checkpoint sau mỗi epoch, và cell tính `test_accuracy`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q /content/drive/MyDrive/archive\ \(13\).zip -d /content/archive\ \(13\)/

## 1. Cài thư viện

Chạy cell này trên Colab nếu môi trường chưa có đủ thư viện. Nếu chạy local đã cài sẵn thì có thể bỏ qua.


In [ ]:
%pip -q install torch torchvision pandas pillow tqdm matplotlib torchmetrics


In [ ]:
!wget -c 'https://dinov3.llamameta.net/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiNTFma3Jxa3l3bTl3MGpweHI2NndvNWZsIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3ODkyOTc0MTJ9fX1dfQ__&Signature=sdowk%7EiiF6j8599sB9K5-M3sB%7Elq1be-oKP%7El7VBISEmjHEBPyCWw9Y2JD4oJY9LbXE313Q4ZgjX%7EXO3EiwGQZ3MZhfkZr0OXmADX8L%7E7XX%7E4mkVA1JSOkVzEa3Oj39gY%7E%7ER21%7EmJXEcFOZPMj%7EzGs-uxIh1xeRrfo-ZUqv3Yz7z7vUgn1OoDhu98kPnh8KT4MF1Gd6gcB4LogqWk3ooVScR6z8ECw50eawfY8lyUUzv%7E%7Em1QdstSU1k6HFhKFbuuogf0MyH7rzYScskxtGG95ke47CKn6CWSNbHLKhcLDfNx-XrN0A-xxis10rpJaMh2jBmxwohSr8XQlGVCBPWqQ__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1749739516259745' \
  -O /content/dinov3_vits16_pretrain_lvd1689m-08c60483.pth

## 2. Import, cấu hình, dataset và candidate pool

Cell này đọc standard train/test split từ `archive (13)`. Không tách validation. Candidate pool được sample cố định từ 20% train split bằng `SEED`; context pair sẽ được tạo ở cell DINO similarity retrieval sau khi load DINOv3 frozen.


In [ ]:
import json
import random
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

PROJECT_DIR = Path.cwd()
DATASET_ROOT_CANDIDATES = [
    PROJECT_DIR / 'archive (13)',
    Path('F:/Documents/CODE/Python/cv_project/XLA/archive (13)'),
    Path('/content/archive (13)'),
    Path('/content/drive/MyDrive/archive (13)'),
]
DATASET_ROOT = next(
    (p for p in DATASET_ROOT_CANDIDATES if (p / 'anno_train.csv').is_file() and (p / 'anno_test.csv').is_file() and (p / 'names.csv').is_file()),
    DATASET_ROOT_CANDIDATES[0],
)
TRAIN_IMAGE_ROOT = DATASET_ROOT / 'car_data' / 'car_data' / 'train'
TEST_IMAGE_ROOT = DATASET_ROOT / 'car_data' / 'car_data' / 'test'

OUT_PUT_DIR = PROJECT_DIR / 'outputs' / 'baseline_frozen_dino_similarity_ret_cars'
OUTPUT_DIR = OUT_PUT_DIR
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

DINOV3_REPO_URL = 'https://github.com/facebookresearch/dinov3.git'
DINOV3_REPO_DIR = str(PROJECT_DIR / 'dinov3')
DINOV3_MODEL_NAME = 'dinov3_vits16'
DINOV3_WEIGHTS_URL = 'https://dl.fbaipublicfiles.com/dinov3/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
DINOV3_WEIGHTS = str(PROJECT_DIR / 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth')
BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 384

IMAGE_SIZE = 256
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
NUM_CLASSES = 196
TOP_K = 5
USE_BBOX_CROP = False
CANDIDATE_POOL_FRACTION = 0.20
RETRIEVAL_BATCH_SIZE = 128
RETRIEVAL_CHUNK_SIZE = 1024

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomResizedCrop((IMAGE_SIZE, IMAGE_SIZE), scale=(0.75, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])


def load_class_names(dataset_root):
    class_names = pd.read_csv(dataset_root / 'names.csv', header=None)[0].astype(str).tolist()
    if len(class_names) != NUM_CLASSES:
        raise ValueError(f'Expected {NUM_CLASSES} classes, got {len(class_names)}')
    return class_names


def load_annotations(dataset_root, split):
    df = pd.read_csv(
        dataset_root / f'anno_{split}.csv',
        header=None,
        names=['fname', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'class_id_1based'],
    )
    df['class_id'] = df['class_id_1based'].astype(int) - 1
    df['split'] = split
    df['row_id'] = np.arange(len(df))
    return df


def attach_image_paths(df, image_root):
    image_index = {path.name: path for path in sorted(Path(image_root).rglob('*.jpg'))}
    missing = [fname for fname in df['fname'] if fname not in image_index]
    if missing:
        raise FileNotFoundError(f'Thieu anh trong {image_root}, vi du: {missing[:5]}')
    out = df.copy()
    out['image_path'] = [str(image_index[fname]) for fname in out['fname']]
    return out


class_names = load_class_names(DATASET_ROOT)
train_df = attach_image_paths(load_annotations(DATASET_ROOT, 'train'), TRAIN_IMAGE_ROOT).reset_index(drop=True)
test_df = attach_image_paths(load_annotations(DATASET_ROOT, 'test'), TEST_IMAGE_ROOT).reset_index(drop=True)
NUM_CLASSES = len(class_names)

candidate_rng = np.random.default_rng(SEED)
candidate_pool_size = max(1, int(round(len(train_df) * CANDIDATE_POOL_FRACTION)))
candidate_pool_positions = np.sort(candidate_rng.choice(len(train_df), size=candidate_pool_size, replace=False))
candidate_df = train_df.iloc[candidate_pool_positions].reset_index(drop=True)
candidate_df.to_csv(OUT_PUT_DIR / 'candidate_pool.csv', index=False)

print('DATASET_ROOT:', DATASET_ROOT.resolve())
print('TRAIN_IMAGE_ROOT:', TRAIN_IMAGE_ROOT.resolve())
print('TEST_IMAGE_ROOT:', TEST_IMAGE_ROOT.resolve())
print('OUT_PUT_DIR:', OUT_PUT_DIR.resolve())
print(f'Standard train images: {len(train_df):,}')
print(f'Standard test images: {len(test_df):,}')
print(f'Candidate pool: {len(candidate_df):,}/{len(train_df):,} ({CANDIDATE_POOL_FRACTION:.0%})')
print(f'Train label range: {int(train_df.class_id.min())}..{int(train_df.class_id.max())}')
print(f'Test label range: {int(test_df.class_id.min())}..{int(test_df.class_id.max())}')
display(train_df.head())
display(candidate_df.head())


## 3. Dataset pair và DataLoader

Mỗi sample trả về `(query_image, context_image, label, query_fname, context_fname, context_label, similarity_score)`. Khác Random Context, context ở đây được cố định bởi nearest neighbor trong embedding space của DINOv3 frozen.


In [ ]:
class SimpleImageDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = int(row['bbox_x2'])
            bottom = int(row['bbox_y2'])
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image = self.transform(self._open_image(row))
        return image, int(index), row['fname'], int(row['class_id'])


class StanfordCarsSimilarityContextDataset(Dataset):
    def __init__(self, annotations, candidate_annotations, context_positions, context_scores, query_transform, context_transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.candidate_annotations = candidate_annotations.reset_index(drop=True).copy()
        self.context_positions = [int(x) for x in context_positions]
        self.context_scores = [float(x) for x in context_scores]
        self.query_transform = query_transform
        self.context_transform = context_transform
        self.crop_bbox = crop_bbox
        if len(self.context_positions) != len(self.annotations):
            raise ValueError('context_positions phai co cung do dai voi annotations')
        if len(self.context_scores) != len(self.annotations):
            raise ValueError('context_scores phai co cung do dai voi annotations')

    def __len__(self):
        return len(self.annotations)

    def set_epoch(self, epoch):
        pass

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = int(row['bbox_x2'])
            bottom = int(row['bbox_y2'])
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        query_row = self.annotations.iloc[index]
        context_pos = self.context_positions[index]
        context_row = self.candidate_annotations.iloc[context_pos]
        query_image = self.query_transform(self._open_image(query_row))
        context_image = self.context_transform(self._open_image(context_row))
        label = int(query_row['class_id'])
        context_label = int(context_row['class_id'])
        similarity_score = float(self.context_scores[index])
        return query_image, context_image, label, query_row['fname'], context_row['fname'], context_label, similarity_score


## 4. Frozen DINO similarity retrieval và downstream model

Retriever: DINOv3 ViT-S/16 pretrained được load ở chế độ `eval()` và không update gradient. Notebook tính CLS embedding cho candidate pool, train queries và test queries bằng `eval_transform`, normalize L2, rồi chọn candidate có cosine similarity cao nhất.

Downstream classifier: nhận cặp query-context đã chọn, ưu tiên token-level fusion bằng cách ghép patch tokens của query và context trước transformer blocks. Backbone downstream vẫn được fine-tune bằng AdamW như paper.


In [ ]:
def ensure_dinov3_assets():
    repo_dir = Path(DINOV3_REPO_DIR)
    weights_path = Path(DINOV3_WEIGHTS)
    if not repo_dir.is_dir():
        try:
            print('Cloning DINOv3 repo to:', repo_dir)
            subprocess.run(['git', 'clone', '--depth', '1', DINOV3_REPO_URL, str(repo_dir)], check=True)
        except Exception as exc:
            print('Khong clone duoc DINOv3 repo. torch.hub se thu tai tu GitHub khi tao model.')
            print(type(exc).__name__ + ':', exc)
    else:
        print('DINOv3 repo da ton tai:', repo_dir)
    if not weights_path.is_file():
        print('Dang tai DINOv3 ViT-S/16 weights ve:', weights_path)
        urlretrieve(DINOV3_WEIGHTS_URL, weights_path)
    else:
        print('DINOv3 weights da ton tai:', weights_path)
    if weights_path.stat().st_size < 10 * 1024 * 1024:
        raise ValueError(f'File weight co ve bi loi hoac qua nho: {weights_path}')
    print('Weights size MB:', round(weights_path.stat().st_size / 1024**2, 2))


def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)
    if not path.is_file():
        raise FileNotFoundError(f'Khong tim thay file: {path}')
    if path.stat().st_size < 10 * 1024 * 1024:
        raise ValueError('File weight bi rong hoac bi hong')
    return str(path)


def load_dinov3_backbone():
    repo_dir = Path(DINOV3_REPO_DIR)
    source = 'local' if repo_dir.is_dir() else 'github'
    repo_or_dir = str(repo_dir) if repo_dir.is_dir() else 'facebookresearch/dinov3'
    return torch.hub.load(repo_or_dir, DINOV3_MODEL_NAME, source=source, weights=resolve_dinov3_weights())


def extract_cls_features(backbone, images):
    features = backbone(images)
    if isinstance(features, dict):
        if 'x_norm_clstoken' in features:
            return features['x_norm_clstoken']
        if 'x_norm_cls_token' in features:
            return features['x_norm_cls_token']
        raise KeyError(f'Khong tim thay CLS token trong output keys: {list(features)}')
    return features


def extract_frozen_dino_embeddings(annotations, backbone, desc):
    dataset = SimpleImageDataset(annotations, eval_transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    embeddings = []
    backbone.eval()
    for parameter in backbone.parameters():
        parameter.requires_grad_(False)
    with torch.inference_mode():
        for images, *_ in tqdm(loader, desc=desc, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                features = extract_cls_features(backbone, images)
            features = F.normalize(features.float(), dim=1)
            embeddings.append(features.cpu())
    return torch.cat(embeddings, dim=0)


def retrieve_nearest_contexts(query_embeddings, candidate_embeddings, query_df, candidate_df, exclude_self=False, chunk_size=1024):
    candidate_embeddings = candidate_embeddings.float()
    selected_positions = []
    selected_scores = []
    candidate_row_id_to_position = {int(row_id): pos for pos, row_id in enumerate(candidate_df['row_id'].tolist())}

    for start in tqdm(range(0, len(query_embeddings), chunk_size), desc='retrieve nearest context', leave=False):
        end = min(start + chunk_size, len(query_embeddings))
        similarities = query_embeddings[start:end].float() @ candidate_embeddings.T

        if exclude_self:
            for local_index, query_index in enumerate(range(start, end)):
                query_row_id = int(query_df.iloc[query_index]['row_id'])
                context_pos = candidate_row_id_to_position.get(query_row_id)
                if context_pos is not None and len(candidate_df) > 1:
                    similarities[local_index, context_pos] = -float('inf')

        scores, positions = similarities.max(dim=1)
        selected_positions.extend([int(x) for x in positions.tolist()])
        selected_scores.extend([float(x) for x in scores.tolist()])

    return selected_positions, selected_scores


def build_similarity_pairs_df(query_df, candidate_df, context_positions, context_scores):
    rows = []
    for query_pos, (context_pos, score) in enumerate(zip(context_positions, context_scores)):
        query = query_df.iloc[query_pos]
        context = candidate_df.iloc[int(context_pos)]
        rows.append({
            'query_fname': query['fname'],
            'query_split': query['split'],
            'query_class_id': int(query['class_id']),
            'query_car_name': class_names[int(query['class_id'])],
            'context_fname': context['fname'],
            'context_class_id': int(context['class_id']),
            'context_car_name': class_names[int(context['class_id'])],
            'context_pool_position': int(context_pos),
            'context_train_row_id': int(context['row_id']),
            'dino_similarity': float(score),
            'same_class': bool(int(query['class_id']) == int(context['class_id'])),
        })
    return pd.DataFrame(rows)


class DinoV3FrozenSimilarityContextClassifier(nn.Module):
    def __init__(self, repo_dir, model_name, weights, num_classes, feature_dim=384, allow_feature_fallback=True):
        super().__init__()
        repo_dir = Path(repo_dir)
        source = 'local' if repo_dir.is_dir() else 'github'
        repo_or_dir = str(repo_dir) if repo_dir.is_dir() else 'facebookresearch/dinov3'
        self.backbone = torch.hub.load(repo_or_dir, model_name, source=source, weights=weights)
        self.feature_dim = feature_dim
        self.allow_feature_fallback = allow_feature_fallback
        self.supports_token_fusion = all(hasattr(self.backbone, attr) for attr in ['blocks', 'norm']) and (hasattr(self.backbone, 'prepare_tokens_with_masks') or hasattr(self.backbone, 'prepare_tokens'))
        self.active_fusion_mode = 'token_concat' if self.supports_token_fusion else 'feature_fusion'
        self.token_head = nn.Linear(feature_dim, num_classes)
        self.feature_head = nn.Sequential(nn.LayerNorm(feature_dim * 4), nn.Linear(feature_dim * 4, num_classes))

    def _extract_cls(self, features):
        if isinstance(features, dict):
            if 'x_norm_clstoken' in features:
                return features['x_norm_clstoken']
            if 'x_norm_cls_token' in features:
                return features['x_norm_cls_token']
            raise KeyError(f'Khong tim thay CLS token trong output keys: {list(features)}')
        return features

    def _encode_single(self, images):
        return self._extract_cls(self.backbone(images))

    def _prepare_tokens(self, images):
        if hasattr(self.backbone, 'prepare_tokens_with_masks'):
            try:
                return self.backbone.prepare_tokens_with_masks(images, None)
            except TypeError:
                return self.backbone.prepare_tokens_with_masks(images)
        if hasattr(self.backbone, 'prepare_tokens'):
            return self.backbone.prepare_tokens(images)
        raise AttributeError('Backbone khong expose prepare_tokens API')

    def _forward_token_concat(self, query_images, context_images):
        query_tokens = self._prepare_tokens(query_images)
        context_tokens = self._prepare_tokens(context_images)
        num_register_tokens = int(getattr(self.backbone, 'num_register_tokens', 0))
        num_prefix_tokens = 1 + num_register_tokens
        pair_tokens = torch.cat([query_tokens[:, :num_prefix_tokens], query_tokens[:, num_prefix_tokens:], context_tokens[:, num_prefix_tokens:]], dim=1)
        for block in self.backbone.blocks:
            pair_tokens = block(pair_tokens)
        pair_tokens = self.backbone.norm(pair_tokens)
        return self.token_head(pair_tokens[:, 0])

    def _forward_feature_fusion(self, query_images, context_images):
        q = self._encode_single(query_images)
        c = self._encode_single(context_images)
        pair = torch.cat([q, c, torch.abs(q - c), q * c], dim=1)
        return self.feature_head(pair)

    def forward(self, query_images, context_images):
        if self.supports_token_fusion:
            try:
                self.active_fusion_mode = 'token_concat'
                return self._forward_token_concat(query_images, context_images)
            except (AttributeError, TypeError, KeyError) as exc:
                if not self.allow_feature_fallback:
                    raise
                self.supports_token_fusion = False
                self.active_fusion_mode = 'feature_fusion'
                print('Token-level fusion khong kha dung, fallback sang feature fusion:', type(exc).__name__, exc)
        self.active_fusion_mode = 'feature_fusion'
        return self._forward_feature_fusion(query_images, context_images)


ensure_dinov3_assets()

retriever_backbone = load_dinov3_backbone().to(DEVICE).eval()
candidate_embeddings = extract_frozen_dino_embeddings(candidate_df, retriever_backbone, 'candidate embeddings')
train_query_embeddings = extract_frozen_dino_embeddings(train_df, retriever_backbone, 'train query embeddings')
test_query_embeddings = extract_frozen_dino_embeddings(test_df, retriever_backbone, 'test query embeddings')

del retriever_backbone
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

train_context_positions, train_context_scores = retrieve_nearest_contexts(
    train_query_embeddings,
    candidate_embeddings,
    train_df,
    candidate_df,
    exclude_self=True,
    chunk_size=RETRIEVAL_CHUNK_SIZE,
)
test_context_positions, test_context_scores = retrieve_nearest_contexts(
    test_query_embeddings,
    candidate_embeddings,
    test_df,
    candidate_df,
    exclude_self=False,
    chunk_size=RETRIEVAL_CHUNK_SIZE,
)

train_similarity_pairs_df = build_similarity_pairs_df(train_df, candidate_df, train_context_positions, train_context_scores)
test_similarity_pairs_df = build_similarity_pairs_df(test_df, candidate_df, test_context_positions, test_context_scores)
train_similarity_pairs_df.to_csv(OUT_PUT_DIR / 'train_frozen_dino_similarity_pairs.csv', index=False)
test_similarity_pairs_df.to_csv(OUT_PUT_DIR / 'test_frozen_dino_similarity_pairs.csv', index=False)
torch.save({
    'candidate_embeddings': candidate_embeddings,
    'train_query_embeddings': train_query_embeddings,
    'test_query_embeddings': test_query_embeddings,
    'candidate_pool_positions': torch.as_tensor(candidate_pool_positions, dtype=torch.long),
    'train_context_positions': torch.as_tensor(train_context_positions, dtype=torch.long),
    'train_context_scores': torch.as_tensor(train_context_scores, dtype=torch.float32),
    'test_context_positions': torch.as_tensor(test_context_positions, dtype=torch.long),
    'test_context_scores': torch.as_tensor(test_context_scores, dtype=torch.float32),
    'seed': SEED,
    'candidate_pool_fraction': CANDIDATE_POOL_FRACTION,
}, OUT_PUT_DIR / 'frozen_dino_retrieval_cache.pt')

train_dataset = StanfordCarsSimilarityContextDataset(train_df, candidate_df, train_context_positions, train_context_scores, train_transform, train_transform, crop_bbox=USE_BBOX_CROP)
test_dataset = StanfordCarsSimilarityContextDataset(test_df, candidate_df, test_context_positions, test_context_scores, eval_transform, eval_transform, crop_bbox=USE_BBOX_CROP)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

model = DinoV3FrozenSimilarityContextClassifier(DINOV3_REPO_DIR, DINOV3_MODEL_NAME, resolve_dinov3_weights(), NUM_CLASSES, FEATURE_DIM).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')
checkpoint_path = OUT_PUT_DIR / 'final_frozen_dino_similarity_ret_dinov3_vits16_cars.pt'
last_checkpoint_path = OUT_PUT_DIR / 'checkpoint_last.pt'

sample = train_dataset[0]
print('Saved retrieval cache:', (OUT_PUT_DIR / 'frozen_dino_retrieval_cache.pt').resolve())
print('Saved train similarity pairs:', (OUT_PUT_DIR / 'train_frozen_dino_similarity_pairs.csv').resolve())
print('Saved test similarity pairs:', (OUT_PUT_DIR / 'test_frozen_dino_similarity_pairs.csv').resolve())
print('Sample tensors:', sample[0].shape, sample[1].shape)
print('Query/context/similarity:', sample[3], '->', sample[4], sample[6])
print('Initial fusion mode:', model.active_fusion_mode)
display(test_similarity_pairs_df.head())
print(model)


## 5. Train và lưu checkpoint sau từng epoch

Sau mỗi epoch, notebook lưu ngay `checkpoint_last.pt`. File này có cả model, optimizer, scheduler, history, candidate pool và similarity retrieval pairs để resume hoặc kiểm tra lại.


In [ ]:
def train_one_epoch(network, loader, epoch):
    network.train()
    total_loss = 0.0
    all_targets = []
    all_predictions = []
    progress = tqdm(loader, desc=f'train epoch {epoch}', leave=False)
    for query_images, context_images, labels, *_ in progress:
        query_images = query_images.to(DEVICE, non_blocking=True)
        context_images = context_images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            logits = network(query_images, context_images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * query_images.size(0)
        predictions = logits.argmax(dim=1)
        all_targets.extend(labels.detach().cpu().tolist())
        all_predictions.extend(predictions.detach().cpu().tolist())
        progress.set_postfix(loss=f'{loss.item():.4f}', fusion=network.active_fusion_mode)
    return {'loss': total_loss / len(loader.dataset), 'accuracy': float(np.mean(np.asarray(all_targets) == np.asarray(all_predictions)))}


def build_checkpoint_payload(epoch, history):
    return {
        'epoch': int(epoch),
        'method': 'Frozen DINO Similarity Retrieval',
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': list(history),
        'class_names': class_names,
        'architecture': BACKBONE_NAME,
        'feature_dim': FEATURE_DIM,
        'fusion_mode': getattr(model, 'active_fusion_mode', 'unknown'),
        'dinov3_model_name': DINOV3_MODEL_NAME,
        'dinov3_repo_dir': str(DINOV3_REPO_DIR),
        'dinov3_weights': str(resolve_dinov3_weights()),
        'dataset': 'Stanford Cars',
        'dataset_root': str(DATASET_ROOT),
        'standard_train_size': len(train_dataset),
        'standard_test_size': len(test_dataset),
        'image_size': IMAGE_SIZE,
        'mean': imagenet_mean,
        'std': imagenet_std,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'seed': SEED,
        'use_bbox_crop': USE_BBOX_CROP,
        'candidate_pool_fraction': CANDIDATE_POOL_FRACTION,
        'candidate_pool_size': len(candidate_df),
        'candidate_pool_positions': [int(x) for x in candidate_pool_positions],
        'train_context_positions': [int(x) for x in train_context_positions],
        'train_context_scores': [float(x) for x in train_context_scores],
        'test_context_positions': [int(x) for x in test_context_positions],
        'test_context_scores': [float(x) for x in test_context_scores],
        'retriever': 'frozen DINOv3 cosine similarity on CLS embeddings',
        'pretraining': 'DINOv3',
    }


RESUME_FROM_LAST_CHECKPOINT = False
history = []
start_epoch = 1
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

if RESUME_FROM_LAST_CHECKPOINT and last_checkpoint_path.is_file():
    saved_checkpoint = torch.load(last_checkpoint_path, map_location=DEVICE)
    model.load_state_dict(saved_checkpoint['model_state_dict'])
    if 'optimizer_state_dict' in saved_checkpoint:
        optimizer.load_state_dict(saved_checkpoint['optimizer_state_dict'])
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(DEVICE)
    if 'scheduler_state_dict' in saved_checkpoint:
        scheduler.load_state_dict(saved_checkpoint['scheduler_state_dict'])
    history = saved_checkpoint.get('history', [])
    start_epoch = int(saved_checkpoint.get('epoch', 0)) + 1
    print(f'Resumed from epoch {start_epoch - 1}: {last_checkpoint_path.resolve()}')

for epoch in range(start_epoch, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, epoch)
    scheduler.step()
    history.append({
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_accuracy': train_metrics['accuracy'],
        'learning_rate': scheduler.get_last_lr()[0],
        'fusion_mode': getattr(model, 'active_fusion_mode', 'unknown'),
    })
    history_df = pd.DataFrame(history)
    history_df.to_csv(OUT_PUT_DIR / 'training_history.csv', index=False)
    torch.save(build_checkpoint_payload(epoch, history), last_checkpoint_path)
    print(f"Epoch {epoch:03d}/{EPOCHS} | train loss {train_metrics['loss']:.4f} | train top-1 accuracy {train_metrics['accuracy']:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | fusion {getattr(model, 'active_fusion_mode', 'unknown')}")
    print(f'Saved latest checkpoint: {last_checkpoint_path.resolve()}')

final_epoch = history[-1]['epoch'] if history else 0
torch.save(build_checkpoint_payload(final_epoch, history), checkpoint_path)
history_df = pd.DataFrame(history)
history_df.to_csv(OUT_PUT_DIR / 'training_history.csv', index=False)
print('Saved final checkpoint:', checkpoint_path.resolve())
print('Saved latest checkpoint:', last_checkpoint_path.resolve())


## 6. Vẽ training history

Dùng để kiểm tra loss/accuracy trong quá trình train. Benchmark chính nằm ở cell `Official test accuracy`.


In [ ]:
if 'history_df' not in globals():
    history_df = pd.read_csv(OUT_PUT_DIR / 'training_history.csv')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_df.plot(x='epoch', y='train_loss', ax=axes[0], marker='o', markevery=max(1, len(history_df) // 10), title='Training loss')
history_df.plot(x='epoch', y='train_accuracy', ax=axes[1], marker='o', markevery=max(1, len(history_df) // 10), title='Training top-1 accuracy')
for axis in axes:
    axis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
display(history_df.tail())


## 7. Official test accuracy

Cell này load final checkpoint; nếu final chưa có vì training bị dừng giữa chừng thì dùng `checkpoint_last.pt`. Metric chính là `test_accuracy`, tương ứng hàng **Frozen DINO Similarity Ret.** cho Cars trong Table 2.


In [ ]:
def macro_f1_score_np(y_true, y_pred, num_classes):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    f1_values = []
    for class_id in range(num_classes):
        tp = np.sum((y_true == class_id) & (y_pred == class_id))
        fp = np.sum((y_true != class_id) & (y_pred == class_id))
        fn = np.sum((y_true == class_id) & (y_pred != class_id))
        denom = 2 * tp + fp + fn
        f1_values.append(0.0 if denom == 0 else (2 * tp) / denom)
    return float(np.mean(f1_values))


def evaluate_context_model(network, loader, top_k=TOP_K):
    network.eval()
    total_loss = 0.0
    all_targets = []
    all_predictions = []
    rows = []
    top_k = min(top_k, NUM_CLASSES)
    with torch.inference_mode():
        for query_images, context_images, labels, file_names, context_file_names, context_labels, similarity_scores in tqdm(loader, desc='test', leave=False):
            query_images = query_images.to(DEVICE, non_blocking=True)
            context_images = context_images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = network(query_images, context_images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1).cpu()
            values, indices = torch.topk(probs, k=top_k, dim=1)
            predictions = logits.argmax(dim=1).detach().cpu()
            labels_cpu = labels.detach().cpu()
            total_loss += loss.item() * query_images.size(0)
            all_targets.extend(labels_cpu.tolist())
            all_predictions.extend(predictions.tolist())
            for fname, ctx_fname, ctx_label, sim_score, label, pred, value_row, index_row in zip(file_names, context_file_names, context_labels, similarity_scores, labels_cpu, predictions, values, indices):
                label_id = int(label)
                pred_id = int(pred)
                context_label_id = int(ctx_label)
                top_class_ids = [int(index) for index in index_row]
                rows.append({
                    'fname': fname,
                    'true_class_id': label_id,
                    'true_car_name': class_names[label_id],
                    'context_fname': ctx_fname,
                    'context_class_id': context_label_id,
                    'context_car_name': class_names[context_label_id],
                    'dino_similarity': float(sim_score),
                    'same_class_context': bool(label_id == context_label_id),
                    'predicted_class_id': pred_id,
                    'predicted_car_name': class_names[pred_id],
                    'correct': bool(pred_id == label_id),
                    'confidence': float(value_row[0]),
                    'top_k_class_ids': '|'.join(str(index) for index in top_class_ids),
                    'top_k_car_names': '|'.join(class_names[index] for index in top_class_ids),
                    'top_k_probabilities': '|'.join(f'{float(value):.6f}' for value in value_row),
                })
    y_true = np.asarray(all_targets)
    y_pred = np.asarray(all_predictions)
    return {
        'test_loss': total_loss / len(loader.dataset),
        'test_accuracy': float(np.mean(y_true == y_pred)),
        'test_macro_f1': macro_f1_score_np(y_true, y_pred, NUM_CLASSES),
        'predictions': pd.DataFrame(rows),
    }


evaluation_checkpoint_path = checkpoint_path if checkpoint_path.is_file() else last_checkpoint_path
saved_checkpoint = torch.load(evaluation_checkpoint_path, map_location=DEVICE)
model.load_state_dict(saved_checkpoint['model_state_dict'])
model.eval()
print('Loaded checkpoint for evaluation:', evaluation_checkpoint_path.resolve())

results = evaluate_context_model(model, test_loader, top_k=TOP_K)
test_prediction_df = results.pop('predictions')
test_prediction_df.to_csv(OUT_PUT_DIR / 'test_predictions.csv', index=False)
metrics = {
    'method': 'Frozen DINO Similarity Retrieval',
    'test_accuracy': results['test_accuracy'],
    'test_loss': results['test_loss'],
    'test_macro_f1': results['test_macro_f1'],
    'num_classes': NUM_CLASSES,
    'standard_train_size': len(train_dataset),
    'standard_test_size': len(test_dataset),
    'architecture': BACKBONE_NAME,
    'pretraining': 'DINOv3',
    'retriever': 'frozen DINOv3 cosine similarity on CLS embeddings',
    'fusion_mode': getattr(model, 'active_fusion_mode', saved_checkpoint.get('fusion_mode', 'unknown')),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'seed': SEED,
    'use_bbox_crop': USE_BBOX_CROP,
    'candidate_pool_fraction': CANDIDATE_POOL_FRACTION,
    'candidate_pool_size': len(candidate_df),
    'test_context_same_class_rate': float(test_prediction_df['same_class_context'].mean()),
    'test_mean_dino_similarity': float(test_prediction_df['dino_similarity'].mean()),
    'checkpoint': str(evaluation_checkpoint_path),
}
with open(OUT_PUT_DIR / 'metrics.json', 'w', encoding='utf-8') as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
print(f"test_accuracy: {metrics['test_accuracy']:.4f} ({metrics['test_accuracy'] * 100:.2f}%)")
print(f"test_loss: {metrics['test_loss']:.4f}")
print(f"test_macro_f1: {metrics['test_macro_f1']:.4f}")
print(f"same_class_context_rate: {metrics['test_context_same_class_rate']:.4f}")
print(f"mean_dino_similarity: {metrics['test_mean_dino_similarity']:.4f}")
print('Saved test predictions:', (OUT_PUT_DIR / 'test_predictions.csv').resolve())
print('Saved metrics:', (OUT_PUT_DIR / 'metrics.json').resolve())
display(test_prediction_df.head())


## 8. Inference và load checkpoint

Dùng hàm này để test nhanh một ảnh. Nếu không truyền context thủ công, hàm sẽ dùng DINOv3 frozen để tìm context gần nhất từ candidate pool.


In [ ]:
def find_frozen_dino_context_for_image(image_path):
    image_path = Path(image_path)
    retriever = load_dinov3_backbone().to(DEVICE).eval()
    for parameter in retriever.parameters():
        parameter.requires_grad_(False)
    image = eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            query_embedding = extract_cls_features(retriever, image)
        query_embedding = F.normalize(query_embedding.float(), dim=1).cpu()
    del retriever
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    similarities = query_embedding @ candidate_embeddings.float().T
    score, position = similarities.max(dim=1)
    context_row = candidate_df.iloc[int(position.item())]
    return context_row, float(score.item())


def predict_one_image_with_similar_context(image_path, context_image_path=None, network=model, top_k=TOP_K):
    image_path = Path(image_path)
    if context_image_path is None:
        context_row, similarity_score = find_frozen_dino_context_for_image(image_path)
        context_image_path = Path(context_row['image_path'])
        context_name = class_names[int(context_row['class_id'])]
    else:
        context_image_path = Path(context_image_path)
        context_name = None
        similarity_score = None
    query_image = eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    context_image = eval_transform(Image.open(context_image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    network.eval()
    with torch.inference_mode():
        logits = network(query_image, context_image)
        probs = torch.softmax(logits, dim=1)[0]
        values, indices = torch.topk(probs, k=min(top_k, NUM_CLASSES))
    result = pd.DataFrame({
        'rank': np.arange(1, len(indices) + 1),
        'class_id': [int(index) for index in indices.cpu()],
        'car_name': [class_names[int(index)] for index in indices.cpu()],
        'probability': [float(value) for value in values.cpu()],
    })
    print('Query image:', image_path)
    print('Context image:', context_image_path)
    if context_name is not None:
        print('Context label:', context_name)
    if similarity_score is not None:
        print(f'DINO similarity: {similarity_score:.4f}')
    return result


def load_frozen_dino_similarity_model_for_inference(checkpoint_file, device=DEVICE):
    saved = torch.load(checkpoint_file, map_location=device)
    loaded_model = DinoV3FrozenSimilarityContextClassifier(
        repo_dir=saved.get('dinov3_repo_dir', DINOV3_REPO_DIR),
        model_name=saved.get('dinov3_model_name', saved.get('architecture', DINOV3_MODEL_NAME)),
        weights=saved.get('dinov3_weights', resolve_dinov3_weights()),
        num_classes=len(saved['class_names']),
        feature_dim=saved.get('feature_dim', FEATURE_DIM),
    )
    loaded_model.load_state_dict(saved['model_state_dict'])
    loaded_model.to(device).eval()
    return loaded_model, saved['class_names']

print('Final checkpoint:', checkpoint_path.resolve())
print('Latest checkpoint:', last_checkpoint_path.resolve())
print('Method:', 'Frozen DINO Similarity Retrieval')


## 9. Ghi chú benchmark

- Method này tương ứng hàng **Frozen DINO Similarity Ret.** trong Table 2.
- Candidate pool lấy từ 20% train split.
- Retriever dùng DINOv3 pretrained/frozen, cosine similarity trên normalized CLS embeddings.
- Retriever không học và không dùng test labels để chọn context.
- Downstream classifier được train trên cặp query-context.
- Không dùng validation split và không dùng test để chọn epoch.
- Metric chính cho Cars là `test_accuracy`.
- Những file nên giữ để so sánh method: `metrics.json`, `training_history.csv`, `test_predictions.csv`, `candidate_pool.csv`, `train_frozen_dino_similarity_pairs.csv`, `test_frozen_dino_similarity_pairs.csv`, `frozen_dino_retrieval_cache.pt`, `checkpoint_last.pt`, và final checkpoint.
